In [ ]:
import json

try:
    import yfinance as yf
except ImportError:
    import subprocess
    import sys
    subprocess.check_call([sys.executable, "-m", "pip", "install", "yfinance"])
    import yfinance as yf

dbutils.widgets.text("ticker", "")
dbutils.widgets.text("start_date", "")
dbutils.widgets.text("catalog", "")

ticker = dbutils.widgets.get("ticker")
start_date = dbutils.widgets.get("start_date")
catalog = dbutils.widgets.get("catalog")

from pyspark.sql.functions import expr

In [ ]:
try:
    if start_date:
        prices = yf.download(ticker, start=start_date, interval="1d", auto_adjust=False)
    else:
        prices = yf.download(ticker, period="max", interval="1d", auto_adjust=False)

    prices = prices.reset_index()
    prices.columns = [c.lower().replace(" ", "_") for c in prices.columns]
    max_date = prices["date"].max().strftime("%Y-%m-%d")

    table = ticker.replace(".", "_")
    spark.sql(f"CREATE SCHEMA IF NOT EXISTS {catalog}.yfinance")
    prices = spark.createDataFrame(prices)

    #DQR
    dqr = spark.table(f"{catalog}.config.data_quality_rules")\
        .filter("domain = 'yfinance'")\
        .select("type","column","rule")\
        .collect()

    for r in dqr:
        if r.type == "case":
            prices = prices.withColumn(r.column,expr(r.rule))
        elif r.type == "filter":
            prices = prices.filter(r.rule)


    prices.write.format("delta").mode("append").saveAsTable(
        f"{catalog}.yfinance.`{table}`"
    )

    dbutils.notebook.exit(json.dumps({"ingestionStatus": 1, "last_date": max_date}))
except Exception as e:
    print(f"{ticker} ingestion failed: {e}")
    dbutils.notebook.exit(json.dumps({"ingestionStatus": 0, "last_date": None}))